In [1]:
# !pip install codecarbon
# !pip install -e .

import pandas as pd
import pickle
from trust_library.core import TrustEvaluator
from trust_library.factsheet import (
    load_factsheet_default,
    load_factsheet,
    save_factsheet, 
    create_factsheet_interactive,
    create_factsheet,
)
from trust_library.fairness import statistical_parity_difference

import matplotlib.pyplot as plt
import seaborn as sns
import json

from trust_library.utils import to_json_safe
import plotly.express as px

In [2]:
# USO DE FACTSHEET
# factsheet = create_factsheet_interactive()
# print(load_factsheet_default()) 
# fs = create_factsheet({
# "general": {
#     "target_column": "Target"
# },
# "fairness": {
#     "protected_feature": "Group",
#     "protected_values": [1],      # Valor que identifica al grupo desprotegido
#     "favorable_outcomes": [1]     # Valor que identifica el éxito
# }
# })
# save_factsheet(fs, "factsheet.json")

# USO DE NN
# import torch
# loaded_model = torch.load("model_nn.pth")
# loaded_model.eval()
# class TorchModelWrapper:

#     def __init__(self, model):
#         self.model = model

#     def predict(self, X):
#         X_tensor = torch.tensor(X, dtype=torch.float32)

#         self.model.eval()
#         with torch.no_grad():
#             outputs = self.model(X_tensor)
#             preds = torch.argmax(outputs, dim=1)

#         return preds.numpy()

# USO DE MÉTRICA CONCRETA
# y_pred = loaded_model.predict(
#     test_loaded.drop(
#         columns=[factsheet["general"]["target_column"]["value"]]
#     )
# )

# group_mask = (
#     test_loaded[factsheet["fairness"]["protected_feature"]["value"]]
#     == factsheet["fairness"]["protected_values"]["value"][0]
# )

# y_pred = loaded_model.predict(test_loaded.drop("Target", axis=1))
# group_mask = test_loaded["Group"] == 1

# statistical_parity_difference(y_pred,group_mask)

In [3]:
factsheet = load_factsheet_default() #load_factsheet("factsheet.json")

with open("./models_and_data/model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

train_loaded = pd.read_csv("./models_and_data/train.csv")
test_loaded = pd.read_csv("./models_and_data/test.csv")

evaluator = TrustEvaluator(
    model=loaded_model,
    train_data=train_loaded,
    test_data=test_loaded,
    factsheet=factsheet
)

In [4]:
subset = evaluator.evaluate(show_nan=True)
# Imprimimos subset formateado
print(json.dumps(subset, indent=4))

Computing Fairness metrics...
Metric 'underfitting' computed in 0.00 seconds.
Metric 'overfitting' computed in 0.00 seconds.
Metric 'class_balance' computed in 0.00 seconds.
Metric 'statistical_parity_difference' computed in 0.00 seconds.
Metric 'disparate_impact' computed in 0.00 seconds.
Metric 'equal_opportunity_difference' computed in 0.00 seconds.
Metric 'average_odds_difference' computed in 0.00 seconds.
Metric 'accuracy_parity' computed in 0.00 seconds.
Metric 'predictive_parity' computed in 0.00 seconds.
Metric 'treatment_equality' computed in 0.00 seconds.
Metric 'calibration_gap' computed in 0.01 seconds.
Metric 'well_calibration_error' computed in 0.00 seconds.
Metric 'generalized_entropy_index' computed in 0.00 seconds.
Metric 'theil_index' computed in 0.00 seconds.
Metric 'coefficient_of_variation' computed in 0.00 seconds.
Metric 'individual_consistency' computed in 0.02 seconds.
Metric 'class_imbalance' computed in 0.00 seconds.
Metric 'kl_divergence' computed in 0.00 se

In [5]:
evaluator.plot_results()

In [7]:
evaluator.plot_radar()

In [10]:
with open("./models_and_data/model_SVM.pkl", "rb") as f:
    loaded_model2 = pickle.load(f)
evaluator2 = TrustEvaluator(loaded_model2, train_loaded, test_loaded, factsheet)
subset2 = evaluator2.evaluate(show_nan=True)



Computing Fairness metrics...
Metric 'underfitting' computed in 0.00 seconds.
Metric 'overfitting' computed in 0.00 seconds.
Metric 'class_balance' computed in 0.00 seconds.
Metric 'statistical_parity_difference' computed in 0.00 seconds.
Metric 'disparate_impact' computed in 0.00 seconds.
Metric 'equal_opportunity_difference' computed in 0.00 seconds.
Metric 'average_odds_difference' computed in 0.00 seconds.
Metric 'accuracy_parity' computed in 0.00 seconds.
Metric 'predictive_parity' computed in 0.00 seconds.
Metric 'treatment_equality' computed in 0.00 seconds.
Metric 'calibration_gap' computed in 0.00 seconds.
Metric 'well_calibration_error' computed in 0.00 seconds.
Metric 'generalized_entropy_index' computed in 0.00 seconds.
Metric 'theil_index' computed in 0.00 seconds.
Metric 'coefficient_of_variation' computed in 0.00 seconds.
Metric 'individual_consistency' computed in 0.01 seconds.
Metric 'class_imbalance' computed in 0.00 seconds.
Metric 'kl_divergence' computed in 0.00 se

In [11]:

TrustEvaluator.compare_radar({
    "RandomForest": subset,
    "SVM": subset2
})

TrustEvaluator.compare_all_bars({
    "RandomForest": subset,
    "SVM": subset2
})

In [27]:
import numpy as np


def calculate_score(value: float, thresholds: list[float]) -> int:
    """
    Compute a score from 1 to 5 based on the given value and thresholds.
    Detects whether the thresholds are for ascending (accuracy) or descending 
    (error) metrics and calculates the score accordingly.
    """
    if not thresholds or len(thresholds) == 0:
        raise ValueError("Thresholds must be a non empty list.")
        
    value = abs(value) # Trabajamos siempre con valor absoluto
    
    # Caso: Error (Menor es mejor). Ej: [0.075, 0.05, 0.01, 0]
    # Caso: Accuracy (Mayor es mejor). Ej: [0.8, 0.9, 0.95,0.99]
    idx = np.digitize(value, thresholds, right=False)
    score = idx + 1
        
    # Asegurar que el score nunca exceda 5 ni baje de 1
    return int(np.clip(score, 1, 5))

#(1];(2],(3],(4];(5]
calculate_score(0.075, [0.075, 0.05, 0.01, 0])

4